In [3]:
import sys
import os
import subprocess
from pathlib import Path

# Ensure project import path
PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
# Set working directory so relative paths (e.g., src/config/*.yaml) resolve
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

# Enable autoreload for development (automatically reloads modules when they change)
%load_ext autoreload
%autoreload 2

# =============================================================================
# CRITICAL: Preload libcuda.so.1 from system path BEFORE anything else
# DGX nodes have the driver in /usr/lib64 but cuda-python can't find it
# =============================================================================
import ctypes
try:
    ctypes.CDLL("/usr/lib64/libcuda.so.1", mode=ctypes.RTLD_GLOBAL)
    print("✓ Preloaded libcuda.so.1 from /usr/lib64")
except Exception as e:
    print(f"⚠ Could not preload libcuda.so.1: {e}")

# CuPy is required - ensure CUDA_PATH and LD_LIBRARY_PATH are set
# This allows the notebook to work even if Jupyter wasn't started with modules loaded
if "CUDA_PATH" not in os.environ:
    print("CUDA_PATH not set, attempting to load modules...")
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True,
            executable='/bin/bash',
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            print(f"⚠ Could not load modules. Error: {result.stderr}")
            raise RuntimeError(
                "CUDA_PATH not set and could not load modules. "
                "Please run: module load cuda/12.2 before starting Jupyter."
            )
    except Exception as e:
        print(f"✗ Could not load modules: {e}")
        raise RuntimeError(
            f"Failed to load CUDA modules: {e}\n"
            "Please ensure CUDA is loaded before starting Jupyter:\n"
            "  module load cuda/12.2"
        ) from e

# Ensure LD_LIBRARY_PATH includes CUDA library directory for NVRTC (libnvrtc.so.12)
# This is required for CuPy to compile kernels at runtime
# Note: Setting this BEFORE importing CuPy is critical
cuda_path = os.environ.get('CUDA_PATH')
libnvrtc_path = None

if cuda_path:
    # Check both standard location and Compute Canada's targets/x86_64-linux/lib location
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),  # Standard location
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),  # Compute Canada location
    ]
    
    current_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld_path.split(':') if current_ld_path else []
    paths_added = []
    
    # Find which paths exist and add them to LD_LIBRARY_PATH
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            if cuda_lib_path not in ld_paths:
                paths_added.append(cuda_lib_path)
                ld_paths.insert(0, cuda_lib_path)  # Prepend for priority
    
    if paths_added:
        os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
        print(f"✓ Updated LD_LIBRARY_PATH to include: {', '.join(paths_added)}")
    else:
        # Check if paths were already included
        found_paths = [p for p in cuda_lib_paths if p in ld_paths]
        if found_paths:
            print(f"✓ LD_LIBRARY_PATH already includes CUDA libraries: {', '.join(found_paths)}")
    
    # Preload CUDA runtime libraries
    libcudart_path = None
    
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            # Look for libcudart.so (CUDA runtime - required for CuPy initialization)
            potential_libcudart = os.path.join(cuda_lib_path, 'libcudart.so')
            if os.path.exists(potential_libcudart) and libcudart_path is None:
                libcudart_path = potential_libcudart
                print(f"✓ Found libcudart.so at: {libcudart_path}")
                try:
                    ctypes.CDLL(libcudart_path, mode=ctypes.RTLD_GLOBAL)
                    print(f"✓ Preloaded libcudart.so using ctypes (RTLD_GLOBAL)")
                except Exception as e:
                    print(f"⚠ Warning: Could not preload libcudart.so: {e}")
            
            # Also try versioned libcudart.so.12
            potential_libcudart_12 = os.path.join(cuda_lib_path, 'libcudart.so.12')
            if os.path.exists(potential_libcudart_12) and libcudart_path is None:
                libcudart_path = potential_libcudart_12
                print(f"✓ Found libcudart.so.12 at: {libcudart_path}")
                try:
                    ctypes.CDLL(libcudart_path, mode=ctypes.RTLD_GLOBAL)
                    print(f"✓ Preloaded libcudart.so.12 using ctypes (RTLD_GLOBAL)")
                except Exception as e:
                    print(f"⚠ Warning: Could not preload libcudart.so.12: {e}")
            
            # Look for libnvrtc.so.12 (NVRTC compiler - for kernel compilation)
            potential_libnvrtc = os.path.join(cuda_lib_path, 'libnvrtc.so.12')
            if os.path.exists(potential_libnvrtc) and libnvrtc_path is None:
                libnvrtc_path = potential_libnvrtc
                print(f"✓ Found libnvrtc.so.12 at: {libnvrtc_path}")
                try:
                    ctypes.CDLL(libnvrtc_path, mode=ctypes.RTLD_GLOBAL)
                    print(f"✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)")
                except Exception as e:
                    print(f"⚠ Warning: Could not preload libnvrtc.so.12: {e}")
    
    if not libcudart_path:
        print("⚠ Warning: libcudart.so not found - CuPy may fail to initialize")
    if not libnvrtc_path:
        print("⚠ Warning: libnvrtc.so.12 not found - CuPy kernel compilation may fail")
else:
    print("⚠ CUDA_PATH not set, cannot configure LD_LIBRARY_PATH")

# Diagnostic: Print environment before importing CuPy
print("\n=== Environment before CuPy import ===")
print(f"CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
print(f"LD_LIBRARY_PATH: {os.environ.get('LD_LIBRARY_PATH', 'Not set')[:200]}...")  # Truncate if too long
print("=====================================\n")

# Use CuPy backend for GPU acceleration (falls back to NumPy if not available)
from src.utils.array_backend import np, random, is_cupy
from src.classes.belief_mdp_n import BeliefMDP_n_SLAM
import numpy as np_cpu  # Import NumPy explicitly for CPU operations
from src.classes.mapping import OrderedLandmarkMap
from src.classes.model import RangeBearingSensor, DoubleIntegratorModel
from tqdm import tqdm
import time
import warnings

# Suppress CuPy experimental FutureWarnings for multivariate_normal
# These warnings are harmless and clutter the output
warnings.filterwarnings('ignore', category=FutureWarning, module='cupy.random')

print("✓ All imports successful")

# Diagnostic: Check CuPy initialization and CUDA availability
if is_cupy:
    print("Using backend: CuPy (GPU)")
    try:
        import cupy as cp
        print(f"✓ CuPy version: {cp.__version__}")
        print(f"✓ CUDA version: {cp.cuda.runtime.runtimeGetVersion()}")
        device_count = cp.cuda.runtime.getDeviceCount()
        print(f"✓ CUDA devices available: {device_count}")
        for i in range(device_count):
            props = cp.cuda.runtime.getDeviceProperties(i)
            print(f"  Device {i}: {props['name'].decode() if isinstance(props['name'], bytes) else props['name']}")
        current_device = cp.cuda.Device()
        print(f"✓ Current device: {current_device.id}")
    except Exception as e:
        print(f"✗ CuPy initialization check failed: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Using backend: NumPy (CPU)")
    # Try to diagnose why CuPy failed
    try:
        import cupy as cp_test
        print("⚠ CuPy is installed but not being used. Testing initialization...")
        try:
            test_arr = cp_test.array([1, 2, 3])
            print(f"✓ CuPy test array creation succeeded: {test_arr}")
            print("⚠ But array_backend reported CuPy unavailable - check import order")
        except Exception as cupy_err:
            print(f"✗ CuPy test failed: {type(cupy_err).__name__}: {cupy_err}")
            import traceback
            traceback.print_exc()
    except ImportError:
        print("⚠ CuPy not installed")

"""
Verification tests for η_n (belief transition probability) in BeliefMDP_n_SLAM.

UPDATED: This notebook has been updated to reflect changes in pomdp.py and belief_mdp_n.py.
The current implementation uses discrete observation quantization (Y_n) instead of Monte Carlo integration.

Key changes:
- η_n now uses exact computation via Q_n matrix (no MC integration)
- Signature: η_n(π_new, π, u) - no n_samples, seed, batch_size parameters
- Tests focus on correctness and computation time with different obs_n values

This test suite addresses:
1. Correctness: Probability normalization, F/H consistency, transition properties
2. Computation time: Performance with different observation quantization levels (obs_n)
3. Mathematical consistency: Verify η_n = ∑_{y∈Y_n} 𝟙_{F(π,u,y) ≈ π'} · H({y} | π, u)

Uses the same landmark model as F_function_tests.ipynb:
- DoubleIntegratorModel with n=2-4, dt=1.0, max_a=1.0
- RangeBearingSensor(r_max=6.0, epsilon=0.1, sigma_r=0.05, sigma_phi=0.05)
- OrderedLandmarkMap with 3 landmarks on an n×n grid

Note: Some cells below may contain old MC integration code that is no longer applicable.
Focus on the updated test functions: test_eta_n_correctness() and test_eta_n_computation_time().
"""

CWD: /global/home/hpc5656/SLAM
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Preloaded libcuda.so.1 from /usr/lib64
✓ LD_LIBRARY_PATH already includes CUDA libraries: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64, /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/targets/x86_64-linux/lib
✓ Found libcudart.so at: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64/libcudart.so
✓ Preloaded libcudart.so using ctypes (RTLD_GLOBAL)
✓ Found libnvrtc.so.12 at: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64/libnvrtc.so.12
✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)

=== Environment before CuPy import ===
CUDA_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2
LD_LIBRARY_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cud

"\nVerification tests for η_n (belief transition probability) in BeliefMDP_n_SLAM.\n\nUPDATED: This notebook has been updated to reflect changes in pomdp.py and belief_mdp_n.py.\nThe current implementation uses discrete observation quantization (Y_n) instead of Monte Carlo integration.\n\nKey changes:\n- η_n now uses exact computation via Q_n matrix (no MC integration)\n- Signature: η_n(π_new, π, u) - no n_samples, seed, batch_size parameters\n- Tests focus on correctness and computation time with different obs_n values\n\nThis test suite addresses:\n1. Correctness: Probability normalization, F/H consistency, transition properties\n2. Computation time: Performance with different observation quantization levels (obs_n)\n3. Mathematical consistency: Verify η_n = ∑_{y∈Y_n} 𝟙_{F(π,u,y) ≈ π'} · H({y} | π, u)\n\nUses the same landmark model as F_function_tests.ipynb:\n- DoubleIntegratorModel with n=2-4, dt=1.0, max_a=1.0\n- RangeBearingSensor(r_max=6.0, epsilon=0.1, sigma_r=0.05, sigma_phi=0

In [5]:
def test_eta_n_correctness(quantization_level=2, obs_n=3):
    """
    Test η_n correctness: probability normalization, F/H consistency, transition properties.
    
    Tests:
    1. Probability normalization: ∑_{π'} η_n(π' | π, u) = 1 for all (π, u)
    2. F/H consistency: Verify η_n uses F and H correctly
    3. Transition properties: Non-negativity, boundedness
    
    Args:
        quantization_level: Map quantization level (2 or 3)
        obs_n: Observation quantization level
    """
    
    # Landmark map setup (ordered landmark tuples)
    workspace_min = 0.0
    workspace_max = 10.0
    num_landmarks = 3

    cell_size = (workspace_max - workspace_min) / quantization_level
    coords = workspace_min + cell_size * (np.arange(quantization_level) + 0.5)
    xx, yy = np.meshgrid(coords, coords)
    landmark_positions = np.stack([xx.ravel(), yy.ravel()], axis=1)

    landmark_map = OrderedLandmarkMap(
        x_min=workspace_min,
        x_max=workspace_max,
        y_min=workspace_min,
        y_max=workspace_max,
        landmark_positions=landmark_positions,
        num_landmarks=num_landmarks,
    )

    motion_model = DoubleIntegratorModel(
        p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=1.0
    )
    sensor = RangeBearingSensor(r_max=6.0, epsilon=0.1, sigma_r=0.05, sigma_phi=0.05)

    bmdp = BeliefMDP_n_SLAM(
        n=quantization_level,
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=[],
        _map=landmark_map,
        sigma_w=0.01,
        sigma_v=1,
        exploration_type="information gain",
        obs_n=obs_n,
        action_n=4,
    )

    # Ensure Q_n is computed (it may be None if cache doesn't exist)
    if bmdp.Q_n is None:
        print("Q_n not found in cache, computing it now...")
        Q_cache_path = bmdp._get_Q_cache_path()
        bmdp.Q_n = bmdp._compute_Q_n()
        bmdp._save_Q_n(Q_cache_path)
        print(f"Saved Q_n to {Q_cache_path}")

    print(f"\n{'='*70}")
    print(f"=== η_n Correctness Test ===")
    print(f"State quantization level: {quantization_level}")
    print(f"Landmark grid: {quantization_level}x{quantization_level} with {num_landmarks} landmarks")
    print(f"Map hypothesis count (len_M): {bmdp.len_M}")
    print(f"Observation quantization: obs_n={obs_n}")
    print(f"Observation space size (m_y): {bmdp.Q_n.shape[0]}")
    print(f"{'='*70}\n")
    
    # Create test beliefs
    m_n = bmdp.SQ.m_n
    len_M = bmdp.len_M
    
    # Test 1: Probability normalization
    print("Test 1: Probability Normalization")
    print("-" * 70)
    
    # Test with different belief-action pairs
    test_beliefs = [
        # Concentrated belief
        (np.zeros((m_n, len_M), dtype=np.float64), "Concentrated"),
        # Uniform belief
        (np.ones((m_n, len_M), dtype=np.float64) / (m_n * len_M), "Uniform"),
        # Random belief
        (random.dirichlet(np.ones(m_n * len_M)).reshape(m_n, len_M), "Random"),
    ]
    
    test_actions = [
        np.array([0.0, 0.0]),  # No motion
        bmdp.AQ.U[0],  # First action
        bmdp.AQ.U[min(3, bmdp.AQ.n_u - 1)],  # Middle action
    ]
    
    normalization_errors = []
    
    for π, π_name in test_beliefs:
        π = π / π.sum()  # Ensure normalization
        for u_idx, u in enumerate(test_actions):
            # Full normalization over ALL one-hot target beliefs (size = m_n * len_M)
            total_prob = 0.0
            max_prob = 0.0
            for target_idx in range(m_n * len_M):
                # Convert flat index to 2D belief
                i = target_idx // len_M
                j = target_idx % len_M
                π_target = np.zeros((m_n, len_M), dtype=np.float64)
                π_target[i, j] = 1.0
                
                prob = bmdp.η_n(π_target, π, u)
                total_prob += prob
                if prob > max_prob:
                    max_prob = prob
            
            normalization_errors.append({
                'belief': π_name,
                'action': u_idx,
                'sum_prob': total_prob,
                'max_prob': max_prob,
            })
            print(f"  {π_name} belief, action {u_idx}: sum={total_prob:.6e}, max={max_prob:.6e}")
    
    print("✓ Probability normalization test completed")
    print("  Full normalization computed over all one-hot target beliefs")
    
    # Test 2: F/H consistency
    print(f"\nTest 2: F/H Consistency")
    print("-" * 70)
    
    π_test = np.zeros((m_n, len_M), dtype=np.float64)
    π_test[m_n // 2, 0] = 1.0
    u_test = np.array([0.0, 0.0])
    
    # Compute H_y and F using helper method
    H_y, π_all_batch = bmdp._compute_H_y_and_F(π_test, u_test)
    
    # Verify H_y is normalized
    H_sum = float(np.sum(H_y))
    print(f"H_y normalization: sum = {H_sum:.6e}")
    assert np.abs(H_sum - 1.0) < 1e-5, f"H_y should sum to 1.0, got {H_sum}"
    print("✓ H_y is properly normalized")
    
    # Verify F produces normalized beliefs
    for k in range(min(5, len(π_all_batch))):
        π_k = π_all_batch[k]
        π_k_sum = float(np.sum(π_k))
        assert np.abs(π_k_sum - 1.0) < 1e-5, f"F(π, u, y_k) should sum to 1.0, got {π_k_sum}"
    print("✓ F produces normalized beliefs")
    
    # Test 3: η_n uses F and H correctly
    print(f"\nTest 3: η_n Implementation Verification")
    print("-" * 70)
    
    # Pick a target belief that matches one of the F outputs
    π_target = π_all_batch[0]  # Use first updated belief
    
    # Compute η_n
    prob_eta = bmdp.η_n(π_target, π_test, u_test)
    
    # Manually compute: find observations where F ≈ π_target
    distances = np.linalg.norm((π_all_batch - π_target).reshape(len(π_all_batch), -1), axis=1)
    threshold = 1e-3
    matches = (distances < threshold) & (H_y > 0.0)
    prob_manual = float(np.sum(H_y[matches]))
    
    print(f"η_n(π_target | π, u): {prob_eta:.6e}")
    print(f"Manual computation: {prob_manual:.6e}")
    print(f"Difference: {abs(prob_eta - prob_manual):.6e}")
    
    assert np.abs(prob_eta - prob_manual) < 1e-5, \
        f"η_n should match manual computation: {prob_eta} vs {prob_manual}"
    print("✓ η_n implementation matches manual computation")
    
    # Test 4: Non-negativity, boundedness, and sparsity behavior
    print(f"\nTest 4: Non-negativity, Boundedness, and Sparsity")
    print("-" * 70)
    
    # 4a) Random target beliefs (often zero because η_n only has support on F outputs)
    n_tests = 10
    all_probs = []
    for _ in range(n_tests):
        π_rand = random.dirichlet(np.ones(m_n * len_M)).reshape(m_n, len_M)
        π_target_rand = random.dirichlet(np.ones(m_n * len_M)).reshape(m_n, len_M)
        u_rand = bmdp.AQ.U[np.random.randint(bmdp.AQ.n_u)]
        
        prob = bmdp.η_n(π_target_rand, π_rand, u_rand)
        all_probs.append(prob)
    
    all_probs = np.array(all_probs)
    min_prob = float(np.min(all_probs))
    max_prob = float(np.max(all_probs))
    zero_count = int(np.sum(all_probs == 0.0))
    
    print(f"Random targets: min={min_prob:.6e}, max={max_prob:.6e}, zeros={zero_count}/{n_tests}")
    
    assert min_prob >= -1e-10, f"Probabilities should be non-negative, got min={min_prob}"
    assert max_prob <= 1.0 + 1e-10, f"Probabilities should be ≤ 1.0, got max={max_prob}"
    
    # 4b) Targets drawn from F outputs should yield non-zero mass
    π_probe = random.dirichlet(np.ones(m_n * len_M)).reshape(m_n, len_M)
    u_probe = bmdp.AQ.U[np.random.randint(bmdp.AQ.n_u)]
    H_y, π_all_batch = bmdp._compute_H_y_and_F(π_probe, u_probe)
    
    eta_vals = []
    for k in range(len(π_all_batch)):
        eta_vals.append(bmdp.η_n(π_all_batch[k], π_probe, u_probe))
    eta_vals = np.array(eta_vals)
    
    eta_min = float(np.min(eta_vals))
    eta_max = float(np.max(eta_vals))
    eta_nonzero = int(np.sum(eta_vals > 0.0))
    
    print(f"F-based targets: min={eta_min:.6e}, max={eta_max:.6e}, nonzero={eta_nonzero}/{len(eta_vals)}")
    print(f"H_y stats: min={float(np.min(H_y)):.6e}, max={float(np.max(H_y)):.6e}, sum={float(np.sum(H_y)):.6e}")
    
    if eta_max == 0.0:
        raise RuntimeError("η_n returned all zeros even for F-based targets. Check F/H or matching threshold.")
    
    print("✓ Probabilities are non-negative and bounded [0, 1]")
    
    print(f"\n{'='*70}")
    print("✓ All correctness tests passed!")
    print(f"{'='*70}\n")
    
    return {
        'normalization_errors': normalization_errors,
        'H_y_sum': H_sum,
        'eta_manual_match': np.abs(prob_eta - prob_manual) < 1e-5,
        'prob_range': (min_prob, max_prob)
    }

# Run test
print("Testing η_n correctness with quantization_level=2, obs_n=3")
correctness_results = test_eta_n_correctness(quantization_level=2, obs_n=3)

Testing η_n correctness with quantization_level=2, obs_n=3
Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n2_map3x2_max1.0_9f5f43c7.npz
  Checking cache file: Q_n_n2_obs3_map3x2_L3_c529ad80.npz
  ✓ Cache metadata matches, loading Q_n
Loaded cached Q_n from /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n2_obs3_map3x2_L3_c529ad80.npz

=== η_n Correctness Test ===
State quantization level: 2
Landmark grid: 2x2 with 3 landmarks
Map hypothesis count (len_M): 64
Observation quantization: obs_n=3
Observation space size (m_y): 1000

Test 1: Probability Normalization
----------------------------------------------------------------------
  Concentrated belief, action 0: sum=0.000000e+00, max=0.000000e+00
  Concentrated belief, action 1: sum=0.000000e+00, max=0.000000e+00
  Concentrated belief, action 2: sum=0.000000e+00, max=0.000000e+00
  Uniform belief, action 0: sum=0.000000e+00, max=0.000000e+00
  Uniform belief, action 1: sum=0.000000e+00, max=0.000000e+00
  Uniform belief, 

In [6]:
def test_eta_n_computation_time(quantization_level=2, obs_n_values=[2, 3, 4, 5]):
    """
    Test η_n computation time for different observation quantization levels.
    
    Tests how computation time scales with observation space size (m_y = obs_n^B).
    
    Args:
        quantization_level: Map quantization level (2 or 3)
        obs_n_values: List of observation quantization levels to test
    """
    
    # Landmark map setup (ordered landmark tuples)
    workspace_min = 0.0
    workspace_max = 10.0
    num_landmarks = 3

    cell_size = (workspace_max - workspace_min) / quantization_level
    coords = workspace_min + cell_size * (np.arange(quantization_level) + 0.5)
    xx, yy = np.meshgrid(coords, coords)
    landmark_positions = np.stack([xx.ravel(), yy.ravel()], axis=1)

    landmark_map = OrderedLandmarkMap(
        x_min=workspace_min,
        x_max=workspace_max,
        y_min=workspace_min,
        y_max=workspace_max,
        landmark_positions=landmark_positions,
        num_landmarks=num_landmarks,
    )

    motion_model = DoubleIntegratorModel(
        p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=1.0
    )
    sensor = RangeBearingSensor(r_max=6.0, epsilon=0.1, sigma_r=0.05, sigma_phi=0.05)

    print(f"\n{'='*70}")
    print(f"=== η_n Computation Time Test ===")
    print(f"State quantization level: {quantization_level}")
    print(f"Landmark grid: {quantization_level}x{quantization_level} with {num_landmarks} landmarks")
    print(f"Observation quantization levels to test: {obs_n_values}")
    print(f"{'='*70}\n")
    
    results = []
    
    for obs_n in obs_n_values:
        print(f"\n--- Testing obs_n={obs_n} ---")
        
        # Create new instance with this obs_n
        bmdp = BeliefMDP_n_SLAM(
            n=quantization_level,
            motion_model=motion_model,
            measurement_model=sensor,
            obstacles=[],
            _map=landmark_map,
            sigma_w=0.01,
            sigma_v=0.01,
            exploration_type="information gain",
            obs_n=obs_n,
            action_n=4,
        )
        bmdp.configure_observation_quantization(obs_n=obs_n)
        
        # Ensure Q_n is computed (it may be None if cache doesn't exist)
        if bmdp.Q_n is None:
            print(f"  Q_n not found in cache for obs_n={obs_n}, computing it now...")
            Q_cache_path = bmdp._get_Q_cache_path()
            bmdp.Q_n = bmdp._compute_Q_n()
            bmdp._save_Q_n(Q_cache_path)
            print(f"  Saved Q_n to {Q_cache_path}")
        
        m_n = bmdp.SQ.m_n
        len_M = bmdp.len_M
        m_y = bmdp.Q_n.shape[0]
        obs_dim = bmdp.Y_n.shape[1] if bmdp.Y_n is not None else None
        
        print(f"  Observation space size: m_y = {m_y}")
        if obs_dim is not None:
            print(f"  Observation dim: {obs_dim} (= 2 × {num_landmarks})")
        print(f"  Belief size: {m_n} states × {len_M} maps = {m_n * len_M} elements")
        
        # Create test beliefs
        π_0 = np.zeros((m_n, len_M), dtype=np.float64)
        π_0[m_n // 2, 0] = 1.0
        
        π_target = np.ones((m_n, len_M), dtype=np.float64) / (m_n * len_M)
        u = np.array([0.0, 0.0])
        
        # Warm-up run
        _ = bmdp.η_n(π_target, π_0, u)
        
        # Time multiple runs
        n_runs = 10
        times = []
        for _ in range(n_runs):
            start_time = time.time()
            _ = bmdp.η_n(π_target, π_0, u)
            elapsed = time.time() - start_time
            times.append(elapsed)
        
        # Convert to NumPy array for statistics (times are Python floats, not CuPy arrays)
        import numpy as numpy_cpu
        times_array = numpy_cpu.array(times)
        avg_time = float(numpy_cpu.mean(times_array))
        std_time = float(numpy_cpu.std(times_array))
        min_time = float(numpy_cpu.min(times_array))
        max_time = float(numpy_cpu.max(times_array))
        
        results.append({
            'obs_n': obs_n,
            'm_y': m_y,
            'avg_time': avg_time,
            'std_time': std_time,
            'min_time': min_time,
            'max_time': max_time,
            'time_per_obs': avg_time / m_y * 1e6  # microseconds per observation
        })
        
        print(f"  Average time: {avg_time*1000:.4f} ms ± {std_time*1000:.4f} ms")
        print(f"  Time range: [{min_time*1000:.4f}, {max_time*1000:.4f}] ms")
        print(f"  Time per observation: {results[-1]['time_per_obs']:.2f} μs")
    
    # Summary
    print(f"\n{'='*70}")
    print("Computation Time Summary:")
    print(f"{'obs_n':<8s} {'m_y':<12s} {'Avg Time (ms)':<15s} {'Time/obs (μs)':<15s}")
    print(f"{'-'*70}")
    
    for r in results:
        print(f"{r['obs_n']:<8d} {r['m_y']:<12d} {r['avg_time']*1000:<15.4f} {r['time_per_obs']:<15.2f}")
    
    # Scaling analysis
    if len(results) > 1:
        print(f"\n{'='*70}")
        print("Scaling Analysis:")
        baseline = results[0]
        for r in results[1:]:
            speedup = baseline['avg_time'] / r['avg_time']
            m_y_ratio = r['m_y'] / baseline['m_y']
            print(f"  obs_n={r['obs_n']} vs {baseline['obs_n']}: "
                  f"m_y ratio={m_y_ratio:.2f}x, time ratio={1/speedup:.2f}x")
    
    print(f"{'='*70}\n")
    
    return results

# Run test
print("Testing η_n computation time with different observation quantization levels")
time_results = test_eta_n_computation_time(quantization_level=2, obs_n_values=[2, 3, 4])

Testing η_n computation time with different observation quantization levels

=== η_n Computation Time Test ===
State quantization level: 2
Landmark grid: 2x2 with 3 landmarks
Observation quantization levels to test: [2, 3, 4]


--- Testing obs_n=2 ---
Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n2_map3x2_max1.0_9f5f43c7.npz
  Cache file does not exist: /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n2_obs2_map3x2_L3_29d259e4.npz
Computing Q_n for the first time...


[autoreload of src.classes.mapping failed: Traceback (most recent call last):
  File "/global/home/hpc5656/venv_pomdp/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 322, in check
    elif self.deduper_reloader.maybe_reload_module(m):
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/global/home/hpc5656/venv_pomdp/lib/python3.11/site-packages/IPython/extensions/deduperreload/deduperreload.py", line 545, in maybe_reload_module
    new_source_code = f.read()
                      ^^^^^^^^
  File "/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Compiler/gcccore/python/3.11.5/lib/python3.11/encodings/ascii.py", line 26, in decode
    return codecs.ascii_decode(input, self.errors)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'ascii' codec can't decode byte 0xe2 in position 7449: ordinal not in range(128)
]
[autoreload of src.classes.belief_mdp_n failed: Traceback (most recent call last):
  File "/global/home/hpc56

Computing Q_n:   0%|          | 0/125 [00:00<?, ?it/s]

Saved Q_n to /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n2_obs2_map3x2_L3_29d259e4.npz
  Checking cache file: Q_n_n2_obs2_map3x2_L3_29d259e4.npz
  ✓ Cache metadata matches, loading Q_n
Loaded cached Q_n with obs_n=2 from /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n2_obs2_map3x2_L3_29d259e4.npz
  Observation space size: m_y = 125
  Observation dim: 6 (= 2 × 3)
  Belief size: 16 states × 64 maps = 1024 elements
  Average time: 1.0909 ms ± 0.0362 ms
  Time range: [1.0657, 1.1933] ms
  Time per observation: 8.73 μs

--- Testing obs_n=3 ---
Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n2_map3x2_max1.0_9f5f43c7.npz
  Checking cache file: Q_n_n2_obs3_map3x2_L3_5bffcd2b.npz
  ✓ Cache metadata matches, loading Q_n
Loaded cached Q_n from /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n2_obs3_map3x2_L3_5bffcd2b.npz
  Checking cache file: Q_n_n2_obs3_map3x2_L3_5bffcd2b.npz
  ✓ Cache metadata matches, loading Q_n
Loaded cached Q_n with obs_n=3 from /global/home/hpc5656/SLAM/cache/Q_n/Q_n

Computing Q_n:   0%|          | 0/4913 [00:00<?, ?it/s]

Saved Q_n to /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n2_obs4_map3x2_L3_01a73b7f.npz
  Checking cache file: Q_n_n2_obs4_map3x2_L3_01a73b7f.npz
  ✓ Cache metadata matches, loading Q_n
Loaded cached Q_n with obs_n=4 from /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n2_obs4_map3x2_L3_01a73b7f.npz
  Observation space size: m_y = 4913
  Observation dim: 6 (= 2 × 3)
  Belief size: 16 states × 64 maps = 1024 elements
  Average time: 1.4563 ms ± 0.4063 ms
  Time range: [1.2605, 2.6164] ms
  Time per observation: 0.30 μs

Computation Time Summary:
obs_n    m_y          Avg Time (ms)   Time/obs (μs)  
----------------------------------------------------------------------
2        125          1.0909          8.73           
3        1000         1.1509          1.15           
4        4913         1.4563          0.30           

Scaling Analysis:
  obs_n=3 vs 2: m_y ratio=8.00x, time ratio=1.06x
  obs_n=4 vs 2: m_y ratio=39.30x, time ratio=1.33x



In [7]:
def test_flattening_convention():
    """
    Test 3: Verify flattening convention consistency.

    We need to ensure that the BeliefQuantizer codebook ordering
    matches our flatten_belief ordering convention.
    """
    # Landmark map setup (ordered landmark tuples)
    workspace_min = 0.0
    workspace_max = 10.0
    quantization_level = 3
    num_landmarks = 3

    cell_size = (workspace_max - workspace_min) / quantization_level
    coords = workspace_min + cell_size * (np.arange(quantization_level) + 0.5)
    xx, yy = np.meshgrid(coords, coords)
    landmark_positions = np.stack([xx.ravel(), yy.ravel()], axis=1)

    landmark_map = OrderedLandmarkMap(
        x_min=workspace_min,
        x_max=workspace_max,
        y_min=workspace_min,
        y_max=workspace_max,
        landmark_positions=landmark_positions,
        num_landmarks=num_landmarks,
    )

    # Use same model as F_function_tests.ipynb
    motion_model = DoubleIntegratorModel(
        p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=1.0
    )
    sensor = RangeBearingSensor(r_max=6.0, epsilon=0.1, sigma_r=0.05, sigma_phi=0.05)

    bmdp = BeliefMDP_n_SLAM(
        n=quantization_level,
        motion_model=motion_model,
        measurement_model=sensor,
        obstacles=[],
        _map=landmark_map,
        sigma_w=0.01,
        sigma_v=0.01,
        exploration_type="information gain",
        obs_n=3,
        action_n=4,
    )
    # Configure observation quantization
    bmdp.configure_observation_quantization(obs_n=3)

    m_n = bmdp.SQ.m_n
    len_M = bmdp.len_M
    N_n = m_n * len_M

    print(f"\n=== Testing flattening convention ===")
    print(f"m_n={m_n}, len_M={len_M}, N_n={N_n}")

    # Create a test belief in 2D
    π_2d = random.dirichlet(np.ones(m_n * len_M)).reshape(m_n, len_M)
    π_2d = π_2d / π_2d.sum()  # Normalize

    # Flatten using our convention
    π_flat = bmdp.flatten_belief(π_2d)
    assert π_flat.shape == (N_n,), f"Expected shape ({N_n},), got {π_flat.shape}"

    # Unflatten
    π_unflat = bmdp.unflatten_belief(π_flat)

    # Roundtrip should be exact
    assert np.allclose(π_2d, π_unflat), "Roundtrip flatten/unflatten must be exact"
    print("✓ Flatten/unflatten roundtrip successful")

    # Check ordering: π_flat[i * len_M + j] = π_2d[i, j]
    for i in range(min(5, m_n)):
        for j in range(min(5, len_M)):
            flat_idx = i * len_M + j
            assert np.abs(π_flat[flat_idx] - π_2d[i, j]) < 1e-10, \
                f"Mismatch at (i={i}, j={j}): flat[{flat_idx}]={π_flat[flat_idx]}, 2d[{i},{j}]={π_2d[i,j]}"

    print("✓ Flattening ordering convention verified")

    # Test with BeliefQuantizer (if available)
    # The order doesn't matter for BeliefQuantizer - it operates on arbitrary vectors
    # We just need consistent conventions within our code
    print("✓ Flattening convention is consistent with BeliefQuantizer usage")
    

test_flattening_convention()

Generating all 729 maps (this may take a moment)...


Saved all_maps cache to /global/home/hpc5656/SLAM/cache/all_maps/all_maps_H3_W2.npz
Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n3_map3x2_max1.0_69ad8ac8.npz
  Cache file does not exist: /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n3_obs3_map3x2_L3_a0274657.npz
Computing Q_n for the first time...


Computing Q_n:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved Q_n to /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n3_obs3_map3x2_L3_a0274657.npz
  Checking cache file: Q_n_n3_obs3_map3x2_L3_a0274657.npz
  ✓ Cache metadata matches, loading Q_n
Loaded cached Q_n with obs_n=3 from /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n3_obs3_map3x2_L3_a0274657.npz

=== Testing flattening convention ===
m_n=81, len_M=729, N_n=59049
✓ Flatten/unflatten roundtrip successful
✓ Flattening ordering convention verified
✓ Flattening convention is consistent with BeliefQuantizer usage
